In [ ]:
import sys
import json
import time
import logging
from pathlib import Path
import requests
import os

_PROJECT_ROOT = Path.cwd().resolve()
if _PROJECT_ROOT.name == "notebooks":
    _PROJECT_ROOT = _PROJECT_ROOT.parent.parent
elif _PROJECT_ROOT.name == "podscan":
    _PROJECT_ROOT = _PROJECT_ROOT.parent
sys.path.insert(0, str(_PROJECT_ROOT))

from google_utils.google_sheet import GoogleSheetService
from data.constants import CREDENTIALS_FILE

# Setup logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

In [ ]:
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1pPF2clctk6NxAfI5AjgUJhasy8YrT1b_9XEf6MGbJe0/edit?gid=1458675702#gid=1458675702"

# Column layout: X = guest_entity_id, Y = guest_entity_details
ENTITY_ID_COL = "X"
ENTITY_DETAILS_COL = "Y"
ENTITY_ID_COL_INDEX = 23  # 0-based, X
ENTITY_DETAILS_COL_INDEX = 24  # 0-based, Y
SKIP_SHEET = "List Info"

# Rate limiting and batching (Podscan free tier: 5 req/min)
REQUESTS_PER_MINUTE = 5
API_DELAY_SECONDS = 60 / REQUESTS_PER_MINUTE  # 12s between calls to stay under 5/min
WRITE_BATCH_SIZE = 5  # flush to Google Sheets every N updates (same as podscan_guest_entity_extractor)
SKIP_IF_ALREADY_FILLED = True  # skip rows where guest_entity_details is non-empty

# Load Podscan API key (set PODSCAN_API_KEY env var)
PODSCAN_API_KEY = os.getenv("PODSCAN_API_KEY")
if not PODSCAN_API_KEY:
    logger.warning("PODSCAN_API_KEY not set. Set it before running the main loop.")

In [ ]:
PODSCAN_API_BASE = "https://podscan.fm/api/v1"

def fetch_entity(
    entity_id: str,
    api_key: str,
    with_appearances: bool = False,
    appearances_limit: int = 10,
) -> dict | None:
    """
    Fetch entity details from Podscan API.
    GET /api/v1/entities/{entityId}
    Returns the full response JSON (entity object) or None on error.
    """
    if not entity_id or not str(entity_id).strip():
        return None
    if not api_key or not str(api_key).strip():
        logger.error("Podscan API key is required")
        return None

    url = f"{PODSCAN_API_BASE}/entities/{entity_id.strip()}"
    headers = {"Authorization": f"Bearer {api_key}"}
    params = {}
    if with_appearances:
        params["with_appearances"] = "true"
        params["appearances_limit"] = min(appearances_limit, 50)

    def _do_request():
        return requests.get(url, headers=headers, params=params if params else None, timeout=30)

    try:
        resp = _do_request()
        if resp.status_code == 200:
            return resp.json()
        if resp.status_code == 401:
            logger.error("Podscan API: 401 Unauthorized - check API key")
            return None
        if resp.status_code == 404:
            logger.warning(f"Podscan API: 404 Not found for entity {entity_id}")
            return None
        if resp.status_code == 429:
            retry_after = 60  # wait full minute before retry
            logger.warning(f"Podscan API: 429 Rate limited. Waiting {retry_after}s before retry...")
            time.sleep(retry_after)
            resp = _do_request()
            if resp.status_code == 200:
                return resp.json()
            if resp.status_code == 429:
                logger.warning("Podscan API: 429 on retry - skipping")
                return None
        logger.warning(f"Podscan API: {resp.status_code} for entity {entity_id}: {resp.text[:200]}")
        return None
    except requests.RequestException as e:
        logger.warning(f"Podscan API network error for {entity_id}: {e}")
        return None


def index_to_column_letter(idx: int) -> str:
    """Convert 0-based column index to Google Sheets letter (A=0, B=1, ..., Z=25, AA=26, ...)."""
    result = ""
    idx += 1
    while idx > 0:
        idx -= 1
        result = chr(65 + idx % 26) + result
        idx //= 26
    return result


def parse_entity_to_guest_columns(entity_data: dict) -> dict:
    """
    Parse entity JSON into guest_* column values.
    Returns dict: {guest_names, guest_companies, guest_occupations, guest_industries,
                   guest_twitter, guest_linkedin, guest_instagram, guest_website, guest_other_social}
    """
    result = {
        "guest_names": "",
        "guest_companies": "",
        "guest_occupations": "",
        "guest_industries": "",
        "guest_twitter": "",
        "guest_linkedin": "",
        "guest_instagram": "",
        "guest_website": "",
        "guest_other_social": "",
    }
    if not entity_data or not isinstance(entity_data, dict):
        return result

    result["guest_names"] = str(entity_data.get("entity_name") or "").strip()
    result["guest_companies"] = str(entity_data.get("company") or "").strip()
    result["guest_occupations"] = str(entity_data.get("occupation") or "").strip()
    result["guest_industries"] = str(entity_data.get("industry") or "").strip()
    result["guest_website"] = str(entity_data.get("url") or "").strip()

    social_links = entity_data.get("social_links")
    other_urls = []
    if isinstance(social_links, list):
        for link in social_links:
            if not isinstance(link, dict):
                continue
            url = str(link.get("url") or "").strip()
            if not url:
                continue
            url_lower = url.lower()
            if "instagram.com" in url_lower:
                result["guest_instagram"] = url
            elif "linkedin.com" in url_lower:
                result["guest_linkedin"] = url
            elif "twitter.com" in url_lower or "x.com" in url_lower:
                result["guest_twitter"] = url
            else:
                other_urls.append(url)
    result["guest_other_social"] = ", ".join(other_urls) if other_urls else ""

    return result

In [ ]:
sheet_service = GoogleSheetService(credentials_file=CREDENTIALS_FILE)
spreadsheet_id = sheet_service.extract_spreadsheet_id(SPREADSHEET_URL)

success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
if success:
    sheets_to_process = [s for s in sheet_names if s != SKIP_SHEET]
    print(f"Spreadsheet ID: {spreadsheet_id}")
    print(f"All sheets: {sheet_names}")
    print(f"Sheets to process (excluding '{SKIP_SHEET}'): {sheets_to_process}")
else:
    print(f"Error listing sheets: {sheet_names}")

In [ ]:
def flush_updates(updates: list) -> int:
    """Write accumulated updates to the sheet via batchUpdate. Returns count written."""
    if not updates:
        return 0
    sheet_service.service.spreadsheets().values().batchUpdate(
        spreadsheetId=spreadsheet_id,
        body={"valueInputOption": "RAW", "data": list(updates)},
    ).execute()
    count = len(updates)
    updates.clear()
    return count


def run_entity_details_enrichment():
    """Main loop: read guest_entity_id from col X, fetch from Podscan API, write JSON to col Y."""
    if not PODSCAN_API_KEY:
        logger.error("Set PODSCAN_API_KEY environment variable and re-run")
        return

    success, sheet_names = sheet_service.list_sheets(spreadsheet_id)
    if not success or not isinstance(sheet_names, list):
        logger.error(f"Cannot list sheets: {sheet_names}")
        return

    sheets_to_process = [s for s in sheet_names if s != SKIP_SHEET]
    logger.info(f"Processing {len(sheets_to_process)} sheets (excluding '{SKIP_SHEET}')")

    for sheet_name in sheets_to_process:
        logger.info("=" * 60)
        logger.info(f"Sheet: {sheet_name}")
        logger.info("=" * 60)

        try:
            success, data = sheet_service.get_sheet_values(
                spreadsheet_id, f"'{sheet_name}'!A:Z"
            )
            if not success:
                logger.error(f"  Error fetching data: {data}")
                continue
            if not data:
                logger.info("  No data, skipping")
                continue

            headers = data[0]
            # Resolve column indices by header if present; else use fixed indices
            try:
                entity_id_idx = headers.index("guest_entity_id") if "guest_entity_id" in headers else ENTITY_ID_COL_INDEX
                details_idx = headers.index("guest_entity_details") if "guest_entity_details" in headers else ENTITY_DETAILS_COL_INDEX
            except (ValueError, AttributeError):
                entity_id_idx = ENTITY_ID_COL_INDEX
                details_idx = ENTITY_DETAILS_COL_INDEX

            # Resolve guest column headers -> column letters (skip if header doesn't exist)
            GUEST_HEADERS = [
                "guest_names", "guest_companies", "guest_occupations", "guest_industries",
                "guest_twitter", "guest_linkedin", "guest_instagram", "guest_website", "guest_other_social",
            ]
            guest_col_map = {}
            for h in GUEST_HEADERS:
                if h in headers:
                    idx = headers.index(h)
                    guest_col_map[h] = index_to_column_letter(idx)
                else:
                    guest_col_map[h] = None

            updates = []
            rows_in_batch = 0
            total_written = 0
            processed = 0
            skipped_empty = 0
            skipped_filled = 0
            failed = 0

            for i, row in enumerate(data[1:], start=2):
                guest_entity_id = (row[entity_id_idx] if entity_id_idx < len(row) else "").strip()
                existing_details = (row[details_idx] if details_idx < len(row) else "").strip()

                if not guest_entity_id:
                    skipped_empty += 1
                    continue
                if SKIP_IF_ALREADY_FILLED and existing_details:
                    skipped_filled += 1
                    continue

                response = fetch_entity(guest_entity_id, PODSCAN_API_KEY)
                if response is None:
                    failed += 1
                    logger.info(f"  [row {i}] {guest_entity_id} -> fetch failed")
                else:
                    entity_data = response.get("entity", response)
                    json_str = json.dumps(entity_data, default=str, indent=None)
                    updates.append({
                        "range": f"'{sheet_name}'!{ENTITY_DETAILS_COL}{i}",
                        "values": [[json_str]],
                    })
                    parsed = parse_entity_to_guest_columns(entity_data)
                    for col_header, col_letter in guest_col_map.items():
                        if col_letter:
                            val = parsed.get(col_header, "") or ""
                            updates.append({
                                "range": f"'{sheet_name}'!{col_letter}{i}",
                                "values": [[val]],
                            })
                    processed += 1
                    rows_in_batch += 1
                    logger.info(f"  [row {i}] {guest_entity_id} -> ok")

                if rows_in_batch >= WRITE_BATCH_SIZE:
                    written = flush_updates(updates)
                    total_written += written
                    rows_in_batch = 0
                    logger.info(f"  ** Flushed batch of {written} cells (total written: {total_written})")

                time.sleep(API_DELAY_SECONDS)

            if updates:
                written = flush_updates(updates)
                total_written += written
                logger.info(f"  ** Flushed final batch of {written} (total written: {total_written})")

            logger.info(
                f"  Summary: processed={processed} | failed={failed} | "
                f"skipped_empty={skipped_empty} | skipped_filled={skipped_filled} | written={total_written}"
            )

        except Exception as e:
            logger.exception(f"  ERROR on sheet '{sheet_name}': {e}")

    logger.info("All sheets processed.")

In [ ]:
run_entity_details_enrichment()